# Spatial Downscaling Demo (MMPS + DDIM)

## Initialization

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append("..")  # repo root, so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import torch
from azula.guidance import MMPSDenoiser
from azula.sample import DDIMSampler
from einops import rearrange
from functools import partial

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from cartopy.util import add_cyclic_point
from mpl_toolkits.axes_grid1 import make_axes_locatable
import ipywidgets as widgets
from IPython.display import display

from src.datasets.era5 import ERA5Dataset
from src.utils.era5 import get_era5_area_weighting, load_stats
from src.utils.utils import load_class
from src.utils.plotting import infer_era5_colormap

torch.set_float32_matmul_precision("high")

REPO_ROOT = Path("..").resolve()


## Configuration

Point `CHECKPOINT` at a release checkpoint. 
A current checkpoint can be found here: https://ml.jku.at/research/WIND/downloads/wind_1p5deg.ckpt


In [ ]:
CHECKPOINT = "SET_THIS"  # <-- set this
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# --- downscaling / sampling settings ---
DOWN_SIZE = (60, 120)  # (h, w) of the low-resolution observation
VAR_Y = 0.0015  # observation noise variance
MMPS_ITERATIONS = 2
DDIM_STEPS = 15
DDIM_ETA = 0.0
N_ENSEMBLES = 5
USE_AREA_WEIGHTS = False  # weight pixels by latitude-dependent area in the A operator

# --- data: pick the exact date and time you want to downscale ---
START_DATETIME = "2021-06-15T12:00"  # must match an available analysis hour

ckpt_path = CHECKPOINT
print(f"Using checkpoint: {ckpt_path}")

## Load model

In [ ]:
raw_ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
if "release_meta" not in raw_ckpt:
    raise ValueError(
        f"{ckpt_path} has no 'release_meta' -- it looks like a raw training checkpoint, "
        "not one produced by scripts/prepare_release_checkpoint.py"
    )
release_meta = raw_ckpt["release_meta"]
del raw_ckpt

local_stats_path = (
    REPO_ROOT / "src" / "datasets" / "stats" / release_meta["stats_filename"]
)
assert local_stats_path.exists(), f"Stats file not found at {local_stats_path}"

ModelClass = load_class(release_meta["model_target"])
model = ModelClass.load_from_checkpoint(
    checkpoint_path=ckpt_path,
    map_location="cpu",
    weights_only=False,
    stats_path=str(local_stats_path),  # override the training machine's absolute path
)
model.load_ema_weights()  # forecast outputs look wrong otherwise (see CLAUDE.md debugging rules)
model.freeze()
model.eval()
model.to(DEVICE)

N_TIMESTEPS = release_meta["n_timesteps"]
print(f"max_tokens / n_timesteps = {N_TIMESTEPS}")


## Build a single sample for the requested date/time

In [ ]:
data_fields = release_meta["data"]["fields"]
hours = tuple(release_meta["data"]["hours_val"])
stride = release_meta["data"]["stride_val"]
add_day_year_progress = release_meta["data"]["add_day_year_progress"]
time_chunk_size = release_meta["data"]["time_chunk_size"]

# Same local/GCS fallback as `ERA5DataModule` (see configs/data/era5_1p5deg.yaml) -- set
# ERA5_1P5DEG_PATH to use the local zarr store, otherwise this falls back to GCS.
data_dir_local = os.environ.get("ERA5_1P5DEG_PATH")
data_dir_global = (
    "gs://weatherbench2/datasets/era5/"
    "1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr"
)
if data_dir_local and (
    os.path.exists(data_dir_local) or data_dir_local.startswith("gs://")
):
    data_dir = data_dir_local
else:
    data_dir = data_dir_global
print(f"Using data_dir: {data_dir}")

stats = load_stats(str(local_stats_path), data_fields)
mean, std = stats["mean"], stats["std"]

start = pd.Timestamp(START_DATETIME)
# buffer so the window contains at least N_TIMESTEPS samples from `start` on. NOTE:
# samples are spaced 24 / len(hours) * stride hours apart (12-hourly here), not
# 6-hourly, so this buffer is only just sufficient rather than generous.
end = start + pd.Timedelta(hours=6 * stride * (N_TIMESTEPS + 4))

dataset = ERA5Dataset(
    data_dir=data_dir,
    split="val",
    n_timesteps=N_TIMESTEPS,
    fields=data_fields,
    hours=hours,
    stride=stride,
    timespan=(str(start.date()), str(end.date())),
    mean=mean,
    std=std,
    add_day_year_progress=add_day_year_progress,
    time_chunk_size=time_chunk_size,
)
dataset._ensure_open()

idx = int(np.searchsorted(dataset._time_values, np.datetime64(start)))
assert dataset._time_values[idx] == np.datetime64(start), (
    f"{start} is not an available analysis time. Available hours: {hours}, "
    f"nearest times: {dataset._time_values[max(idx - 1, 0) : idx + 2]}"
)

sample = dataset[idx]
print("target_fields:", sample["target_fields"].shape)
print("additional_inputs:", sample["model_kwargs"]["additional_inputs"].shape)


In [ ]:
x0 = sample["target_fields"].unsqueeze(0).to(DEVICE)  # (1, T, C, H, W)
model_kwargs = {
    "additional_inputs": sample["model_kwargs"]["additional_inputs"]
    .unsqueeze(0)
    .to(DEVICE)
}


## Build the downscaling operator `A`

Average-pooling operator, optionally weighted by grid-cell area so it matches an area-weighted MSE metric.

Note: with 121 latitude rows and a pooling kernel/stride of 2, the last (pole) latitude row
is dropped by `avg_pool2d`, so the low-resolution observation never constrains that row.
The low-res coordinates used for plotting further below are correspondingly approximate.


In [ ]:
def A(x, down_size=DOWN_SIZE):
    b, t, c, h, w = x.shape
    x = rearrange(x, "b t c h w -> (b t) c h w")
    kernel_h, kernel_w = h // down_size[0], w // down_size[1]
    x = torch.nn.functional.avg_pool2d(
        x, kernel_size=(kernel_h, kernel_w), stride=(kernel_h, kernel_w)
    )
    x = rearrange(x, "(b t) c h w -> b t c h w", b=b)
    return x.flatten(-4)


def A_area_weighted(x, area_weights, down_size=DOWN_SIZE):
    b, t, c, h, w = x.shape
    x = rearrange(x, "b t c h w -> (b t) c h w")
    kernel_h, kernel_w = h // down_size[0], w // down_size[1]
    w_map = area_weights.to(x.device).unsqueeze(0).unsqueeze(0)
    numerator = torch.nn.functional.avg_pool2d(
        x * w_map, kernel_size=(kernel_h, kernel_w), stride=(kernel_h, kernel_w)
    )
    denominator = torch.nn.functional.avg_pool2d(
        w_map, kernel_size=(kernel_h, kernel_w), stride=(kernel_h, kernel_w)
    )
    x = numerator / denominator
    x = rearrange(x, "(b t) c h w -> b t c h w", b=b)
    return x.flatten(-4)


if USE_AREA_WEIGHTS:
    orig_h = x0.shape[-2]
    resolution = 180.0 / (orig_h - 1)
    area_weights = get_era5_area_weighting(resolution=resolution).to(DEVICE)
    A_fn = partial(A_area_weighted, area_weights=area_weights, down_size=DOWN_SIZE)
else:
    A_fn = partial(A, down_size=DOWN_SIZE)


## Run MMPS + DDIM sampling

In [ ]:
y = A_fn(x0)

ensemble_preds = []
with torch.no_grad():
    # TODO: Without loop using batch dim
    for _ in range(N_ENSEMBLES):
        denoiser = MMPSDenoiser(
            model.backbone,
            y=y,
            A=A_fn,
            cov_y=VAR_Y,
            iterations=MMPS_ITERATIONS,
        ).to(DEVICE)
        sampler = DDIMSampler(
            denoiser=denoiser, steps=DDIM_STEPS, eta=DDIM_ETA, device=DEVICE
        )

        x1 = sampler.init(x0.shape)
        x0_hat = sampler(x1, **model_kwargs)
        ensemble_preds.append(x0_hat.cpu())
        del denoiser, sampler, x1, x0_hat

preds = torch.stack(ensemble_preds, dim=0)  # (E, 1, T, C, H, W)
print("preds:", preds.shape)


## Denormalize for plotting

Note: `postprocess` also inverts the log10 transform applied to precipitation fields.
Since `y` was pooled in normalized (log) space, the "Low-res input A(x)" panel shows the
exp of pooled log-precipitation for those fields, not the true area-averaged precipitation.


In [ ]:
C, H, W = x0.shape[2], x0.shape[3], x0.shape[4]

preds_denorm = dataset.postprocess(preds.to(DEVICE)).cpu()  # (E, 1, T, C, H, W)
targets_denorm = dataset.postprocess(x0).cpu()  # (1, T, C, H, W)

y_grid = rearrange(
    y, "b (t c h w) -> b t c h w", t=x0.shape[1], c=C, h=DOWN_SIZE[0], w=DOWN_SIZE[1]
)
y_denorm = dataset.postprocess(y_grid.to(DEVICE)).cpu()  # (1, T, C, down_h, down_w)

field_names = dataset.all_field_names
print(f"{len(field_names)} fields available, e.g.: {field_names[:5]} ...")


## Interactive visualization

Choose a field/level, timestep and ensemble member (or the ensemble mean) to inspect the
low-resolution input, the reconstructed high-resolution field, the ground truth, and the error.


In [ ]:
def _pcolormesh_panel(ax, field, lats, lons, cmap, vmin, vmax, title):
    field_cyc, lons_cyc = add_cyclic_point(field, coord=lons)
    lon2d, lat2d = np.meshgrid(lons_cyc, lats)
    ax.set_global()
    img = ax.pcolormesh(
        lon2d,
        lat2d,
        field_cyc,
        cmap=cmap,
        shading="auto",
        vmin=vmin,
        vmax=vmax,
        transform=ccrs.PlateCarree(),
    )
    ax.coastlines(resolution="110m", linewidth=0.5, color="k", alpha=0.8)
    ax.set_title(title)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.2, axes_class=plt.Axes)
    plt.colorbar(img, cax=cax, orientation="vertical")


def plot_downscaling_result(field_name, t, ensemble):
    c = field_names.index(field_name)
    lats, lons = dataset.latitude, dataset.longitude

    target_field = targets_denorm[0, t, c].numpy()
    low_res_field = y_denorm[0, t, c].numpy()
    if ensemble == "ensemble mean":
        pred_field = preds_denorm[:, 0, t, c].mean(dim=0).numpy()
    else:
        pred_field = preds_denorm[int(ensemble.split()[-1]), 0, t, c].numpy()
    error_field = pred_field - target_field

    cmap = infer_era5_colormap(field_name)
    vmin = min(target_field.min(), pred_field.min(), low_res_field.min())
    vmax = max(target_field.max(), pred_field.max(), low_res_field.max())
    err_abs = np.abs(error_field).max()

    fig, axes = plt.subplots(
        1, 4, figsize=(20, 4), subplot_kw=dict(projection=ccrs.PlateCarree())
    )
    low_res_lats = np.linspace(lats.min(), lats.max(), low_res_field.shape[0])
    low_res_lons = np.linspace(
        lons.min(), lons.max(), low_res_field.shape[1], endpoint=False
    )

    _pcolormesh_panel(
        axes[0],
        low_res_field,
        low_res_lats,
        low_res_lons,
        cmap,
        vmin,
        vmax,
        "Low-res input A(x)",
    )
    _pcolormesh_panel(
        axes[1],
        pred_field,
        lats,
        lons,
        cmap,
        vmin,
        vmax,
        f"Reconstruction ({ensemble})",
    )
    _pcolormesh_panel(
        axes[2], target_field, lats, lons, cmap, vmin, vmax, "Ground truth"
    )
    _pcolormesh_panel(
        axes[3],
        error_field,
        lats,
        lons,
        "coolwarm",
        -err_abs,
        err_abs,
        "Error (pred - truth)",
    )

    fig.suptitle(
        f"{field_name} — t={t} ({pd.Timestamp(sample['time_s'][t], unit='s')})"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


widgets.interact(
    plot_downscaling_result,
    field_name=widgets.Dropdown(options=field_names, description="Field:"),
    t=widgets.IntSlider(min=0, max=N_TIMESTEPS - 1, step=1, description="Timestep:"),
    ensemble=widgets.Dropdown(
        options=["ensemble mean"] + [f"member {i}" for i in range(N_ENSEMBLES)],
        description="Ensemble:",
    ),
)
